In [ ]:
pip install findspark

In [ ]:
import socket

server_host = 'HYPE' # Or the IP address of HYPE
server_port = 1521

try:
    with socket.create_connection((server_host, server_port), timeout=5):
        print(f"Successfully connected to {server_host} on port {server_port}.")
except socket.error as e:
    print(f"Connection failed: {e}")
    print("Please check the SQL Server configuration and firewall settings on the server machine.")


In [1]:
import os
import findspark

# --- Set the SPARK_HOME environment variable manually ---
# Use forward slashes or double backslashes in the path
os.environ["SPARK_HOME"] = "C:/spark" 

findspark.init()
print("Findspark initialized successfully.") # Added confirmation message

Findspark initialized successfully.


In [ ]:
# dont run this cell temporarily
'''
import os
import findspark
from pyspark.sql import SparkSession
import pyspark

# --- Set the SPARK_HOME environment variable manually ---
os.environ["SPARK_HOME"] = "C:/spark" 

findspark.init()
print("Findspark initialized successfully.") 

# Define the paths to your local JAR files
mssql_jar_path = r"file:///C:/spark_drivers/mssql-jdbc-13.2.1.jre11.jar"
# Assuming ojdbc17.jar is for an Oracle DB you might use later, adding it here.
#osql_jar_path = r"file:///C:/spark_drivers/ojdbc17.jar"

# --- MongoDB Configuration Variables ---
MONGO_HOST = "localhost"
MONGO_PORT = "27017"
MONGO_DB = "AdventureSales"
MONGO_COLLECTION = "customerdist_1910a"
MONGO_CONNECTOR_PACKAGE = "org.mongodb.spark:mongo-spark-connector_2.12:3.0.2"

# --- Initialize Spark Session with ALL required JARs and Packages ---
spark = SparkSession.builder \
    .appName("SQLServerToosqlTransfer") \
    .config("spark.jars", f"{mssql_jar_path}") \
    .config("spark.jars.packages", MONGO_CONNECTOR_PACKAGE) \
    .config("spark.mongodb.output.uri", f"mongodb://{MONGO_HOST}:{MONGO_PORT}/{MONGO_DB}.{MONGO_COLLECTION}") \
    .getOrCreate()

print("Spark session created with MongoDB and SQL Server support.")
'''

In [2]:
import os
import findspark
from pyspark.sql import SparkSession

# --- Setup for SQL Read Only ---
os.environ["SPARK_HOME"] = "C:/spark" 
findspark.init()
print("Findspark initialized successfully.") 
mssql_jar_path = r"file:///C:/spark_drivers/mssql-jdbc-13.2.1.jre11.jar"
spark = SparkSession.builder \
    .appName("SQLServerReadOnly") \
    .config("spark.jars", mssql_jar_path) \
    .getOrCreate()
print("Spark session created with only SQL Server support.")

Findspark initialized successfully.
Spark session created with only SQL Server support.


In [3]:

# --- SQL Connection Details ---
SQL_TABLE_NAME = "customerdist_1910a"
jdbc_url = r"jdbc:sqlserver://HYPE\SQL_2022:1433;databaseName=sales;encrypt=true;trustServerCertificate=true"
PARQUET_OUTPUT_PATH = r"C:/latMongo/customer_data.parquet"

# 1. Read Data from SQL Server
sql_df = spark.read.jdbc(
    url=jdbc_url,
    table=SQL_TABLE_NAME,
    properties={
        "user": "sa",
        "password": "D@1ry$2014",
        "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver",
        "fetchSize": "1000"
    }
) 

print(f"SQL Data loaded successfully. Number of rows: {sql_df.count()}")
sql_df.show(5)

# 2. Save the DataFrame to a local Parquet file (forces a single partition write)
sql_df.coalesce(1).write.mode("overwrite").parquet(PARQUET_OUTPUT_PATH)

print(f"Data saved to {PARQUET_OUTPUT_PATH}")

# Stop the session
spark.stop()


SQL Data loaded successfully. Number of rows: 1237180
+---------------+---------------+--------+-------+--------+--------------------+--------------------+----------+-------+--------------+--------------------+-----------+-----------+---+---+-------+------+--------------------+---------+----------+---------+------+------------+----+
|      branch_id|          sp_id|gtype_id|type_id|gcust_id|            custd_id|          custd_name|   custdid|channel|customer_group|           address_1|sub_channel|type_outlet|crm|gsm|chiller|hanger|            kode_iap|kelurahan|gtype_name|NewOutlet|RetWhs|type_outlet1| Mkt|
+---------------+---------------+--------+-------+--------+--------------------+--------------------+----------+-------+--------------+--------------------+-----------+-----------+---+---+-------+------+--------------------+---------+----------+---------+------+------------+----+
|               |               |        |       |        |                   /|                    |  

In [ ]:
# 2. Write Data to MongoDB Collection (using the 'mongo' format defined in the setup cell)
sql_df.write.format("mongo").mode("overwrite").save()

print(f"Successfully inserted all data into MongoDB collection: {MONGO_COLLECTION}")